# Forest carbon in the U.S. GHG inventory — checking an independent reproduction

This notebook compares **independently produced per-state forest carbon flux estimates**
against the **published Forest Service estimates** for the same year, and reproduces every
headline number and figure in the repository README from the bundled CSV files.

**What this notebook does and does not do.** It reads *finished* per-state estimates from
`data/` and computes the agreement between them and the federal reference. It does **not**
recompute those estimates from Forest Inventory and Analysis (FIA) plot data — the
estimator is a separate piece of software, being prepared for public release as part of
[pyfia](https://github.com/mihiarc/pyfia) in Q1 2027. So "reproduction" here means *we
reproduced the inventory's published numbers independently*, not *this notebook re-runs
our estimator*. The last section explains exactly what pins each estimate to a re-runnable
input.

Everything below needs only `pandas` and `matplotlib`. No database, no credentials.

> **Sign and units — the two ways to get a wrong answer.**
> Our estimates use **positive = sink**; the Forest Service and EPA both publish
> **negative = sink**. Values are **MMT C** (million metric tons of carbon), not CO₂
> equivalent. Also note the bundled Walters uncertainty file's headers read
> "MMT CO2 Eq." but its values are actually MMT C — a mislabel in the source archive,
> so no CO₂→C conversion is applied to it.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


def find_data_dir() -> Path:
    """Locate data/ regardless of where the notebook was launched from.

    Jupyter "Run All" sets the working directory to notebooks/, but nbconvert,
    papermill and some IDEs launch from the repo root, so walk upward looking
    for a data/ directory that actually contains the results file.
    """
    for base in (Path.cwd(), *Path.cwd().parents):
        cand = base / "data"
        if (cand / "forest_carbon_flux_ci.csv").exists():
            return cand
    raise FileNotFoundError(
        "Could not find data/. Keep notebooks/ and data/ as siblings and run "
        "this notebook from inside the repository."
    )


DATA = find_data_dir()
print(f"using data dir: {DATA}")

# Our estimates: positive = sink, MMT C/yr, one row per contiguous state.
ours = pd.read_csv(DATA / "forest_carbon_flux_ci.csv")

# Primary reference: published Forest Service state-level flux for the same year.
# check_names is left at default; the header mislabel is documented in data/README.md.
wal = pd.read_csv(DATA / "walters_2026_frf_net_flux_2023.csv")

# Secondary reference: EPA Annex 3.13 Table A-208, one year earlier (2022).
epa = pd.read_csv(DATA / "epa_a208_2022.csv")

print(f"our estimates: {len(ours)} states, estimator build {ours.forest_carbon_version.iloc[0]}")

## 1. Put both estimates on one sign convention

The reference publishes negative = sink; we report positive = sink. Negating a flux flips
the sign **and** swaps which bound is the upper versus the lower one — forgetting the swap
is the single most common error when combining these tables.

In [ ]:
ref = pd.DataFrame(
    {
        "state_name": wal["State"],
        "ref": -wal["2023 Flux Estimate (MMT CO2 Eq.)"],
        # Upper and lower deliberately cross over here: negating swaps the bounds.
        "ref_lo": -wal["Upper Bound (MMT CO2 Eq.)"],
        "ref_hi": -wal["Lower Bound (MMT CO2 Eq.)"],
    }
)

# Inner join keeps the states present in both. The reference covers more geography
# than we estimate (Alaska, Hawaii, territories), so those rows drop out here.
df = ours.merge(ref, on="state_name", how="inner").rename(
    columns={
        "point_mmt_c": "ours",
        "ci_lower_mmt_c": "ours_lo",
        "ci_upper_mmt_c": "ours_hi",
    }
)

# Two agreement tests, one strict and one coarse.
df["in_band"] = (df.ours >= df.ref_lo) & (df.ours <= df.ref_hi)
df["overlap"] = (df.ours_lo <= df.ref_hi) & (df.ref_lo <= df.ours_hi)

print(f"matched states: {len(df)}")
df[["state", "ours", "ours_lo", "ours_hi", "ref", "ref_lo", "ref_hi", "in_band", "overlap"]].head()

## 2. The headline comparison

Two different questions, and they deserve different weight:

- **Point estimate inside the published band** is the stricter test.
- **Do the two intervals overlap** is deliberately coarse. Two intervals can overlap while
  the point estimates differ substantially, and where the published interval is very wide,
  overlap is close to automatic. Read a high overlap count as *no gross disagreement*,
  not as tight agreement.

In [ ]:
total_ours, total_ref = df.ours.sum(), df.ref.sum()
pct = (total_ours - total_ref) / abs(total_ref) * 100

print(f"{len(df)}-state total, 2023 (positive = sink, MMT C/yr)")
print(f"  this reproduction        : {total_ours:+.1f}")
print(f"  Forest Service published : {total_ref:+.1f}")
print(f"  difference               : {pct:+.1f}%")
print()
print(f"  point estimate inside the published 95% band : {df.in_band.sum()} / {len(df)}")
print(f"  95% intervals overlap                        : {df.overlap.sum()} / {len(df)}")

# Secondary cross-check against EPA's 2022 A-208 table. Different reference YEAR,
# so this is not the headline agreement figure and is never a difference "vs Walters".
epa_ref = pd.DataFrame({"state": epa.state, "epa": -epa.flux_mmt_c_2022})
e = ours.rename(columns={"point_mmt_c": "ours"}).merge(epa_ref, on="state", how="inner")
epa_pct = (e.ours.sum() - e.epa.sum()) / abs(e.epa.sum()) * 100
print()
print(f"  secondary, vs EPA A-208 for 2022: {e.ours.sum():+.1f} vs {e.epa.sum():+.1f} "
      f"= {epa_pct:+.1f}%")
print("  (mostly the one-year reference offset, not an estimator difference)")

## 3. Agreement across all states

Each point is one state. Bars are 95% confidence intervals on **both** axes, because both
quantities are estimates with uncertainty — neither is ground truth. A point on the dashed
line means the two estimates agree exactly.

In [ ]:
# Colours are checked for colour-vision-deficiency separation; see figures/make_figures.py.
OURS, INK = "#2a78d6", "#52514e"

fig, ax = plt.subplots(figsize=(6.4, 6.6), dpi=120)
lim = [min(df.ref_lo.min(), df.ours_lo.min()) - 1.5,
       max(df.ref_hi.max(), df.ours_hi.max()) + 1.5]

ax.plot(lim, lim, ls=(0, (5, 4)), lw=1.0, color=INK, alpha=0.55, label="1:1 (exact agreement)")
ax.errorbar(df.ref, df.ours,
            xerr=[df.ref - df.ref_lo, df.ref_hi - df.ref],
            yerr=[df.ours - df.ours_lo, df.ours_hi - df.ours],
            fmt="none", ecolor=OURS, elinewidth=1.0, alpha=0.32, capsize=0)
# A surface-coloured ring keeps overlapping points readable in the dense cluster.
ax.plot(df.ref, df.ours, "o", ms=5.2, mfc=OURS, mec="white", mew=0.7, ls="none")

ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect("equal")
ax.grid(True, color="#e3e2de", lw=0.6); ax.set_axisbelow(True)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
ax.set_xlabel("Forest Service published estimate (MMT C/yr)", color=INK)
ax.set_ylabel("This reproduction (MMT C/yr)", color=INK)
ax.set_title(f"{len(df)} states · {df.overlap.sum()}/{len(df)} intervals overlap · "
             f"totals differ {pct:+.1f}%", loc="left", fontsize=11)
ax.legend(loc="upper left", frameon=False, fontsize=9)
fig.tight_layout()
plt.show()

## 4. Per-state detail

"What does this say about *my* state?" is the question that matters most to an inventory
team, so here is every state, sorted, with both intervals side by side.

Where our interval is much **narrower** than the published one, that is not evidence we
are more certain. Our uncertainty budget is incomplete — it excludes tree-biomass model
(NSVB) parameter uncertainty, which drives most of the flux — so the widths should be read
as a lower bound.

In [ ]:
REF = "#eb6834"
d = df.sort_values("ours").reset_index(drop=True)
y = range(len(d)); off = 0.19

fig, ax = plt.subplots(figsize=(7.2, 11.5), dpi=120)
ax.axvline(0, color=INK, lw=0.8, alpha=0.45)

for col, colour, label, shift, marker in (
    ("ours", OURS, "This reproduction", off, "o"),
    ("ref", REF, "Forest Service published", -off, "s"),
):
    pos = [v + shift for v in y]
    ax.errorbar(d[col], pos,
                xerr=[d[col] - d[f"{col}_lo"], d[f"{col}_hi"] - d[col]],
                fmt="none", ecolor=colour, elinewidth=1.1, alpha=0.45, capsize=0)
    ax.plot(d[col], pos, marker, ms=4.6, mfc=colour, mec="white", mew=0.6,
            ls="none", label=label)

ax.set_yticks(list(y)); ax.set_yticklabels(d.state, fontsize=8.5)
ax.set_ylim(-0.8, len(d) - 0.2)
ax.grid(True, color="#e3e2de", lw=0.6); ax.set_axisbelow(True)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
ax.set_xlabel("Net flux (MMT C/yr, positive = sink)", color=INK)
ax.set_title(f"Every state with 95% CIs · {df.in_band.sum()}/{len(df)} of our points "
             f"fall inside the published band", loc="left", fontsize=11)
ax.legend(loc="upper left", frameon=True, fontsize=9)
fig.tight_layout()
plt.show()

Or as a table — the states where our point estimate falls **outside** the published band,
which is where the interesting methodological questions live:

In [ ]:
out = df.loc[~df.in_band, ["state", "ours", "ours_lo", "ours_hi", "ref", "ref_lo", "ref_hi"]]
print(f"{len(out)} of {len(df)} states outside the published band:")
out.sort_values("ours", ascending=False).to_string(index=False)

## 5. Where these numbers came from, and how to re-run them

This notebook deliberately does **not** hide the fact that it read finished estimates from
a file. What makes them checkable rather than merely asserted is that every row records
the inputs that produced it:

- **`evalid`** — the FIA evaluation identifier. This names the exact inventory cycle and
  plot set behind the estimate, so the number is tied to a specific, public, re-runnable
  input rather than to an unnamed snapshot.
- **`n_iter` and `seed`** — the bootstrap iteration count and random seed are fixed and
  recorded, so the confidence interval is reproducible rather than re-randomised on each
  run.
- **`n_pairs`** — how many remeasured plot pairs support the estimate. Flux needs the same
  plot measured twice, so this is well below a state's total plot count, and it is the
  honest denominator for judging how much data stands behind a state's number.

The estimator that consumes those inputs is being prepared for public release as part of
**[pyfia](https://github.com/mihiarc/pyfia)** (MIT-licensed, already on PyPI), targeted
for **Q1 2027**. At that point the pipeline in this table becomes runnable end to end from
public FIA data, and this notebook's comparison becomes a regression test on it rather
than a summary of it.

Until then, the honest description of this repository is: **published results plus the
means to check them against the federal record** — not a runnable estimator.

In [ ]:
# The provenance actually recorded alongside each estimate:
ours[["state", "evalid", "n_pairs", "n_iter", "seed",
      "perturb_litter", "perturb_soc", "forest_carbon_version"]].head(10)